# Temporal Difference Learning: Q-Learning & SARSA

## Abstract

This notebook presents a comprehensive study of Temporal Difference (TD) learning algorithms, focusing on Q-Learning and SARSA for optimal policy control in a GridWorld environment. We explore their mathematical foundations, implementation details, and comparative performance, including an analysis of various exploration strategies. The aim is to demonstrate the practical application of model-free reinforcement learning, providing insights into algorithm selection based on convergence properties, sample efficiency, and safety considerations.

**Keywords:** Temporal Difference Learning, Q-Learning, SARSA, Model-Free Reinforcement Learning, Exploration Strategies, GridWorld, Optimal Control.


## 1. Introduction

Temporal Difference (TD) learning is a core concept in reinforcement learning (RL) that bridges Monte Carlo methods and dynamic programming. Unlike Monte Carlo methods, which require waiting until the end of an episode to compute returns, TD methods update value estimates incrementally after each step. This 'bootstrapping' approach allows for online learning and often results in lower variance updates than Monte Carlo methods.

This assignment delves into two fundamental TD control algorithms: Q-Learning and SARSA. Both aim to learn optimal action-value functions (Q-functions) that guide an agent to maximize cumulative reward in an environment. While Q-Learning is an off-policy algorithm, learning the optimal policy independently of the agent's actual behavior, SARSA is an on-policy algorithm that learns the value of the policy currently being followed, including its exploration strategy. This distinction has important implications for their convergence properties, stability, and suitability for various real-world applications, especially in safety-critical domains.

The project is structured to provide a clear understanding of these algorithms, their practical implementation in a GridWorld environment, and a comparative analysis of their performance characteristics and the impact of different exploration strategies.


## 2. Theoretical Background

Temporal Difference (TD) learning methods are a class of model-free reinforcement learning algorithms that learn directly from experience without requiring a model of the environment's dynamics. They combine ideas from Monte Carlo methods (learning from experience) and dynamic programming (bootstrapping, i.e., updating estimates based on other learned estimates).

### 2.1 TD(0) for Policy Evaluation

TD(0) is the simplest TD algorithm, primarily used for policy evaluation. It learns the state-value function \(V^{\pi}(s)\) for a given policy \(\pi\). Unlike Monte Carlo, which waits until the end of an episode to compute the return, TD(0) performs an update after every single step. The update rule for the value of state \(S_t\) is given by:

\[V(S*t) \leftarrow V(S_t) + \alpha [R*{t+1} + \gamma V(S\_{t+1}) - V(S_t)]\]

Where:

- \(V(S_t)\) is the estimated value of state \(S_t\).
- \(\alpha\) is the learning rate.
- \(R\_{t+1}\) is the immediate reward received after taking action \(A_t\) from \(S_t\).
- \(\gamma\) is the discount factor.
- \(V(S*{t+1})\) is the estimated value of the next state \(S*{t+1}\).
- The term \(R*{t+1} + \gamma V(S*{t+1})\) is the **TD target**.
- The term \(R*{t+1} + \gamma V(S*{t+1}) - V(S_t)\) is the **TD error**, representing the difference between the estimated value and a more accurate one-step lookahead estimate.

### 2.2 Q-Learning: Off-Policy Control

Q-Learning is an off-policy TD control algorithm that learns the optimal action-value function \(Q^_(s,a)\), which represents the maximum expected future reward achievable by taking action \(a\) in state \(s\) and then following the optimal policy thereafter. Being off-policy means it can learn \(Q^_\) while following an exploratory _behavior policy_ (e.g., \(\epsilon\)-greedy), but its updates target the _greedy policy_ (the optimal policy).

The Q-Learning update rule is:

\[Q(S*t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R*{t+1} + \gamma \max*{a'} Q(S*{t+1}, a') - Q(S_t, A_t)]\]

Here:

- \(Q(S_t, A_t)\) is the estimated value of taking action \(A_t\) in state \(S_t\).
- The TD target \(R*{t+1} + \gamma \max*{a'} Q(S*{t+1}, a')\) uses the maximum Q-value of the next state, effectively looking ahead to the best possible action in \(S*{t+1}\), even if the behavior policy did not actually choose that action.

### 2.3 SARSA: On-Policy Control

SARSA (State-Action-Reward-State-Action) is an on-policy TD control algorithm. It learns the action-value function \(Q^{\pi}(s,a)\) for the policy \(\pi\) that the agent is _currently following_, including its exploration strategy. This means that the target value for the update depends on the actual action \(A*{t+1}\) chosen by the behavior policy in the next state \(S*{t+1}\).

The SARSA update rule is:

\[Q(S*t, A_t) \leftarrow Q(S_t, A_t) + \alpha [R*{t+1} + \gamma Q(S*{t+1}, A*{t+1}) - Q(S_t, A_t)]\]

Here:

- The TD target \(R*{t+1} + \gamma Q(S*{t+1}, A*{t+1})\) uses the Q-value of the *next action* \(A*{t+1}\) selected by the _same policy_ that chose \(A_t\). This makes SARSA's learning more conservative and sensitive to the exploration strategy, often performing better in environments where exploration into bad states is costly.

### 2.4 Exploration Strategies

Both Q-Learning and SARSA require an exploration strategy to ensure that all state-action pairs are sufficiently visited to find the optimal policy. Common strategies include:

- **\(\epsilon\)-Greedy Exploration**: With probability \(\epsilon\), a random action is chosen; otherwise, the greedy action (action with the highest Q-value) is selected. \(\epsilon\) typically decays over time to shift from exploration to exploitation.
  \[\pi(a|s) = \begin{cases} 1 - \epsilon + \frac{\epsilon}{|A|} & \text{if } a = \arg\max_a Q(s,a) \\ \frac{\epsilon}{|A|} & \text{otherwise} \end{cases}\]

- **Boltzmann (Softmax) Exploration**: Actions are chosen probabilistically based on their Q-values, with higher Q-values having a higher probability. A temperature parameter \(\tau\) controls the exploration intensity: higher \(\tau\) leads to more uniform probabilities (more exploration), while lower \(\tau\) leads to more greedy choices.
  \[\pi(a|s) = \frac{e^{Q(s,a)/\tau}}{\sum\_{b \in A} e^{Q(s,b)/\tau}}\]

These strategies balance the trade-off between exploiting known good actions and exploring potentially better, unknown actions.


## 3. Methodology

This section outlines the experimental setup, environment description, and implementation architecture employed for studying Temporal Difference learning algorithms. The methodology is designed to systematically evaluate the performance, convergence, and characteristics of TD(0), Q-Learning, and SARSA agents within a controlled reinforcement learning environment.

### 3.1 Environment Description

We utilize a custom-built **GridWorld** environment, which serves as a standard benchmark for tabular reinforcement learning algorithms due to its discrete state and action spaces and clear reward structure. The environment features are:

- **Grid Size**: A \(4 \times 4\) grid, resulting in 16 discrete states.
- **Action Space**: Four deterministic actions: "up", "down", "left", "right".
- **Start State**: Fixed at \((0,0)\) for consistent experimental runs.
- **Goal State**: Located at \((3,3)\), offering a positive reward upon entry.
- **Obstacles**: Specific grid cells \([(1,1), (1,2), (2,1)]\) that incur a penalty if an agent attempts to move into them, causing the agent to remain in its current state.
- **Reward Structure**:
  - Reaching the Goal State: \(+10\)
  - Attempting to move into an Obstacle or out of bounds: \(-5\)
  - All other steps: \(-1\)

### 3.2 Implementation Architecture

The codebase is structured into a modular Python package under the `src/` directory, promoting reusability, readability, and maintainability. The key modules are:

- **`src/config.py`**: Centralizes all hyperparameters (learning rates, discount factors, exploration parameters, environment specifics) to ensure consistency and ease of experimentation.
- **`src/environments.py`**: Contains the `GridWorld` class, responsible for managing state transitions, rewards, and environmental dynamics.
- **`src/agents.py`**: Implements the core TD learning agents (`TD0Agent`, `QLearningAgent`, `SARSAAgent`) along with base classes (`BaseAgent`, `BasePolicy`) and concrete policies (`RandomPolicy`, `GreedyPolicy`).
- **`src/exploration.py`**: Houses various exploration strategies (`epsilon_greedy`, `boltzmann_exploration`) and the `ExplorationExperiment` class for comparative studies.
- **`src/experiments.py`**: Provides high-level functions (`experiment_td0`, `experiment_q_learning`, `experiment_sarsa`) to orchestrate training and result collection for each algorithm.
- **`src/evaluation.py`**: Offers utilities for evaluating agent policies (`evaluate_agent`, `compare_agents`) and analyzing overall performance.
- **`src/visualization.py`**: Contains functions for generating plots and visual representations of learning curves, value functions, and policies (`plot_learning_curve`, `plot_q_learning_analysis`, `compare_algorithms`).
- **`src/utils.py`**: Includes general utilities such as model saving/loading and result exporting (`save_model`, `load_model`, `export_results`).

This modular design facilitates independent development and testing of each component, contributing to a robust and scalable research framework.

### 3.3 Experimental Protocol

For each algorithm and exploration strategy evaluated, a standardized experimental protocol is followed to ensure fair and reproducible comparisons:

1.  **Training Phase**: Each agent is trained for a predefined number of `num_episodes` (e.g., 1000) within the `GridWorld` environment. During training, episode rewards and steps are logged.
2.  **Exploration Schedule**: For Q-Learning and SARSA, an \(\epsilon\)-greedy or Boltzmann exploration strategy is employed, with \(\epsilon\) or \(\tau\) decaying over episodes to encourage a transition from exploration to exploitation.
3.  **Evaluation Phase**: After training, the learned policy of each agent is evaluated over a separate set of `eval_episodes` (e.g., 100) using a purely greedy policy (no exploration). Metrics such as average reward, standard deviation of reward, average steps to goal, and success rate are collected.
4.  **Statistical Analysis**: Results from multiple independent runs (`num_runs`) are collected and averaged to account for stochasticity in the learning process and provide statistically significant performance metrics.
5.  **Visualization**: Key performance indicators, learning curves, value functions, and policies are visualized using `matplotlib` and `seaborn` to provide intuitive insights into the learning dynamics and final learned behaviors.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
import sys
import os

# Add the parent directory of 'src' to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../src')))

# Import modules from the refactored src directory
from src.config import (
    GridWorldConfig,
    AgentConfig,
    ExplorationConfig,
    ExperimentConfig,
    VisualizationConfig,
    SEED,
)
from src.environments import GridWorld
from src.agents import TD0Agent, QLearningAgent, SARSAAgent, RandomPolicy
from src.exploration import ExplorationStrategies, BoltzmannQLearning, ExplorationExperiment
from src.experiments import (
    experiment_td0,
    experiment_q_learning,
    experiment_sarsa,
    experiment_exploration_strategies,
)
from src.evaluation import evaluate_agent, compare_agents, analyze_performance
from src.visualization import (
    plot_learning_curve,
    plot_q_learning_analysis,
    show_q_values,
    compare_algorithms,
)
from src.utils import save_model, load_model, export_results, create_summary_report

# Suppress warnings and set up plotting aesthetics
warnings.filterwarnings("ignore")
np.random.seed(SEED)
random.seed(SEED) # Ensure random module also uses the seed
plt.rcParams["figure.figsize"] = VisualizationConfig.FIGURE_SIZE
plt.rcParams["font.size"] = VisualizationConfig.FONT_SIZE
sns.set_style("whitegrid")

print("✓ Environment setup complete")
print("✓ All modules imported successfully from src/")
print("✓ Ready for temporal difference learning experiments")


In [ ]:
# Initialize the GridWorld environment using configurations from src/config.py
env = GridWorld(
    size=GridWorldConfig.SIZE,
    start_state=GridWorldConfig.START_STATE,
    goal_state=GridWorldConfig.GOAL_STATE,
    obstacles=GridWorldConfig.OBSTACLES,
    step_reward=GridWorldConfig.STEP_REWARD,
    goal_reward=GridWorldConfig.GOAL_REWARD,
    obstacle_reward=GridWorldConfig.OBSTACLE_REWARD,
)

print("GridWorld Environment Configuration:")
print(f"  • State space: {len(env.states)} states")
print(f"  • Action space: {len(env.actions)} actions")
print(f"  • Start state: {env.start_state}")
print(f"  • Goal state: {env.goal_state}")
print(f"  • Obstacles: {env.obstacles}")

# Perform a quick environment reset and step to verify functionality
state = env.reset()
print(f"\nEnvironment reset. Current state: {state}")
next_state, reward, done, info = env.step("right")
print(f"Action 'right': next_state={next_state}, reward={reward}, done={done}")

# Visualize the initial environment layout
env.visualize_values(
    {s: 0 for s in env.states}, # Display all states with value 0 initially
    title=GridWorldConfig.VALUES_TITLE,
    filepath=f"../pictures/gridworld_layout.png"
)
print("✓ Initial GridWorld layout visualization saved to pictures/gridworld_layout.png")


## 4. Implementation and Results

This section details the practical implementation of the TD(0), Q-Learning, and SARSA algorithms, along with the presentation and initial analysis of their learning outcomes. Each algorithm is trained within the GridWorld environment using predefined hyperparameters, and their performance metrics and learned policies are visualized.


### 4.1 TD(0) Policy Evaluation

We begin with TD(0) for policy evaluation. Here, the agent follows a simple `RandomPolicy` to explore the environment, and TD(0) is used to estimate the value function \(V^{\pi}(s)\) for this random policy. This serves as a foundational experiment to understand how value estimates propagate through the state space.


In [ ]:
print("Initializing TD(0) agent...")
random_policy = RandomPolicy(env)
td_agent, V_td = experiment_td0(
    env, random_policy, num_episodes=AgentConfig.NUM_EPISODES, alpha=AgentConfig.ALPHA, gamma=AgentConfig.GAMMA
)

print("\nTD(0) Learning Results:")
env.visualize_values(
    V_td, title="TD(0) Learned Value Function - Random Policy", filepath="../pictures/td0_value_function.png"
)
print("✓ TD(0) value function visualization saved to pictures/td0_value_function.png")

plot_learning_curve(
    td_agent.episode_rewards,
    "TD(0) Policy Evaluation Learning Curve",
    filepath="../pictures/td0_learning_curve.png",
)
print("✓ TD(0) learning curve visualization saved to pictures/td0_learning_curve.png")

key_states = [(0, 0), (1, 0), (2, 0), (3, 2), (2, 2)]
print(f"\nLearned values for key states:")
print("State\t\tTD(0) Value")
print("-" * 30)
for state in key_states:
    if state in V_td:
        print(f"{state}\t\t{V_td[state]:.3f}")
    else:
        print(f"{state}\t\t0.000")

print(f"\n✓ TD(0) policy evaluation completed successfully")


### 4.2 Q-Learning: Off-Policy Control

Next, we implement Q-Learning, an off-policy algorithm designed for optimal control. Q-Learning directly learns the optimal action-value function \(Q^\*(s,a)\) by bootstrapping with the maximum Q-value of the next state. We use an \(\epsilon\)-greedy exploration strategy, with \(\epsilon\) decaying over time, to balance exploration and exploitation.


In [ ]:
print("Initializing Q-Learning agent...")
q_agent, V_optimal, optimal_policy, q_evaluation = experiment_q_learning(
    env, 
    num_episodes=AgentConfig.NUM_EPISODES,
    alpha=AgentConfig.ALPHA,
    gamma=AgentConfig.GAMMA,
    epsilon=ExplorationConfig.EPSILON_START,
    epsilon_decay=ExplorationConfig.EPSILON_DECAY,
    epsilon_min=ExplorationConfig.EPSILON_MIN,
)

print("\nQ-Learning Results:")
env.visualize_values(
    V_optimal, 
    title="Q-Learning: Optimal Value Function V*", 
    policy=optimal_policy,
    filepath="../pictures/q_learning_optimal_value_function.png"
)
print("✓ Q-Learning optimal value function visualization saved to pictures/q_learning_optimal_value_function.png")

print("\nEvaluating learned optimal policy...")
print(f"Policy Evaluation Results:")
print(
    f"  • Average reward: {q_evaluation['avg_reward']:.2f} ± {q_evaluation['std_reward']:.2f}"
)
print(f"  • Average steps to goal: {q_evaluation['avg_steps']:.1f}")
print(f"  • Success rate: {q_evaluation['success_rate']*100:.1f}%")

plot_q_learning_analysis(q_agent, filepath_prefix="q_learning_analysis")
print("✓ Q-Learning analysis plots saved to pictures/q_learning_analysis_*.png")

show_q_values(q_agent)

print("\n✓ Q-Learning successfully learned the optimal policy")
print("✓ Agent demonstrates efficient navigation to goal while avoiding obstacles")


### 4.3 SARSA: On-Policy Control

Finally, we implement SARSA, an on-policy control algorithm. SARSA learns the action-value function for the policy currently being followed, including its exploration steps. This makes SARSA's learning more conservative compared to Q-Learning, which can be beneficial in environments where taking suboptimal exploratory actions is costly or dangerous.


In [ ]:
print("Initializing SARSA agent...")
sarsa_agent, V_sarsa, sarsa_policy, sarsa_evaluation = experiment_sarsa(
    env,
    num_episodes=AgentConfig.NUM_EPISODES,
    alpha=AgentConfig.ALPHA,
    gamma=AgentConfig.GAMMA,
    epsilon=ExplorationConfig.EPSILON_START,
    epsilon_decay=ExplorationConfig.EPSILON_DECAY,
    epsilon_min=ExplorationConfig.EPSILON_MIN,
)

print("\nSARSA Results:")
env.visualize_values(
    V_sarsa, 
    title="SARSA: Learned Value Function", 
    policy=sarsa_policy,
    filepath="../pictures/sarsa_value_function.png"
)
print("✓ SARSA value function visualization saved to pictures/sarsa_value_function.png")

print("\nEvaluating SARSA policy...")
print(f"SARSA Policy Evaluation Results:")
print(
    f"  • Average reward: {sarsa_evaluation['avg_reward']:.2f} ± {sarsa_evaluation['std_reward']:.2f}"
)
print(f"  • Average steps: {sarsa_evaluation['avg_steps']:.1f}")
print(f"  • Success rate: {sarsa_evaluation['success_rate']*100:.1f}%")

plot_q_learning_analysis(sarsa_agent, filepath_prefix="sarsa_analysis")
print("✓ SARSA analysis plots saved to pictures/sarsa_analysis_*.png")

show_q_values(sarsa_agent)

print("\n✓ SARSA successfully learned the on-policy action-value function")
print("✓ Agent demonstrates conservative learning behavior")


### 4.4 Exploration Strategies Analysis

This section investigates the impact of different exploration strategies on the learning performance of Q-Learning agents. We compare fixed \(\epsilon\)-greedy, decaying \(\epsilon\)-greedy, and Boltzmann exploration to understand how they balance the exploration-exploitation dilemma and affect convergence and final performance.


In [ ]:
print("Exploration Strategies Comparison Experiment")
print("=" * 60)

strategies = {
    "epsilon_0.1": {"epsilon": 0.1, "decay": 1.0},
    "epsilon_0.3": {"epsilon": 0.3, "decay": 1.0},
    "epsilon_decay_fast": {"epsilon": 0.9, "decay": 0.99},
    "epsilon_decay_slow": {"epsilon": 0.5, "decay": 0.995},
    "boltzmann_2.0": {"temperature": 2.0, "decay": ExplorationConfig.TEMPERATURE_DECAY, "min": ExplorationConfig.TEMPERATURE_MIN},
}

print("Testing strategies:")
for name, params in strategies.items():
    print(f"  • {name}: {params}")

results_exploration = experiment_exploration_strategies(
    env, strategies, num_episodes=AgentConfig.NUM_EPISODES // 2, num_runs=ExperimentConfig.NUM_RUNS
)

# Analyze and print summary of exploration results
exp_analyzer = ExplorationExperiment(env) # Re-initialize to use its analysis method
performance_analysis = exp_analyzer.analyze_exploration_results(results_exploration)

print("\n" + "=" * 80)
print("EXPLORATION STRATEGY INSIGHTS")
print("=" * 80)
print("1. Fixed epsilon strategies provide consistent exploration")
print("2. Decaying epsilon balances exploration and exploitation over time")
print("3. Boltzmann exploration provides principled probabilistic action selection")
print("4. Higher initial epsilon may find better solutions but converge slower")
print("5. The best strategy depends on environment characteristics")
print("=" * 80)
